# SetFit HTS Classifier Training - Google Colab (Free Tier)

This notebook trains a SetFit model for HTS code classification on Google Colab's free T4 GPU.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** → Save
2. Upload training data from `src/data/`:
   - `crossRulings-train.json`
   - `crossRulings-validation.json`
   - `crossRulings-test.json`

**Expected time:** 30-45 minutes on T4 GPU

> **Note:** For better performance and less chance of OOM crashes, use `colab-train-setfit-optimized.ipynb` with Colab Pro (L4/A100 GPU).

## Step 1: Install Dependencies

In [ ]:
# Pin to known-working versions (transformers 4.57.6 DOES NOT EXIST — was causing install crash)
!pip install -q "setfit==1.1.3" "transformers>=4.44,<5.0" "sentence-transformers>=3.0,<4.0" torch scikit-learn datasets

# Verify install
import setfit, transformers
print(f"✅ setfit=={setfit.__version__}, transformers=={transformers.__version__}")

## Step 2: Upload Training Data

Upload these files from `src/data/`:
- `crossRulings-train.json` (training split)
- `crossRulings-validation.json`
- `crossRulings-test.json`

In [ ]:
from google.colab import files
import json
import os

print("Upload all 3 files from src/data/:")
print("  • crossRulings-train.json")
print("  • crossRulings-validation.json")
print("  • crossRulings-test.json")
print()

uploaded = files.upload()

# Verify
for f in ['crossRulings-train.json', 'crossRulings-validation.json', 'crossRulings-test.json']:
    if os.path.exists(f):
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"✅ {f} ({size_mb:.1f} MB)")
    else:
        print(f"❌ MISSING: {f}")

## Step 3: Prepare Dataset

In [ ]:
import json
from datasets import Dataset
import gc

def load_rulings(filepath):
    import os
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"❌ File not found: {filepath}")
    with open(filepath, 'r') as f:
        data = json.load(f)
        if isinstance(data, list):
            return data
        elif isinstance(data, dict) and 'rulings' in data:
            return data['rulings']
        else:
            raise ValueError(f'Unexpected data format in {filepath}')

def prepare_dataset(rulings, level='subheading', max_samples=None):
    texts = []
    labels = []
    skipped = 0

    for i, ruling in enumerate(rulings):
        if max_samples and i >= max_samples:
            break

        text = ruling.get('productDescription') or ruling.get('product_description', '')
        hts_codes = ruling.get('htsCodes') or ruling.get('hts_codes', [])

        if not text or not hts_codes:
            skipped += 1
            continue

        hts_code = hts_codes[0] if isinstance(hts_codes, list) else hts_codes
        hts_code = hts_code.replace('.', '').replace(' ', '').strip()
        if len(hts_code) < 4:
            skipped += 1
            continue

        if level == 'chapter':
            label = hts_code[:2]
        elif level == 'heading':
            label = hts_code[:4]
        elif level == 'subheading':
            label = hts_code[:6]
        else:
            label = hts_code

        texts.append(text.strip())
        labels.append(label)

    if skipped > 0:
        print(f"  ⚠️  Skipped {skipped} rulings with missing/invalid data")
    return Dataset.from_dict({'text': texts, 'label': labels})

# Load the TRAINING split (not crossRulings.json which is the full dataset)
print("Loading training data...")
train_rulings = load_rulings('crossRulings-train.json')
print(f"Loaded {len(train_rulings)} training rulings")

val_rulings = load_rulings('crossRulings-validation.json')
print(f"Loaded {len(val_rulings)} validation rulings")

test_rulings = load_rulings('crossRulings-test.json')
print(f"Loaded {len(test_rulings)} test rulings")

# T4 has 15GB VRAM — cap at 8K samples to avoid OOM during pair generation
TRAINING_LEVEL = 'heading'
MAX_TRAIN_SAMPLES = 8000

print(f"\nPreparing datasets for '{TRAINING_LEVEL}' classification (max {MAX_TRAIN_SAMPLES} samples)...")
train_dataset = prepare_dataset(train_rulings, level=TRAINING_LEVEL, max_samples=MAX_TRAIN_SAMPLES)
val_dataset = prepare_dataset(val_rulings, level=TRAINING_LEVEL)
test_dataset = prepare_dataset(test_rulings, level=TRAINING_LEVEL)

del train_rulings, val_rulings, test_rulings
gc.collect()

print(f"\nTraining samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Unique labels: {len(set(train_dataset['label']))}")

## Step 4: Train SetFit Model

In [ ]:
from setfit import SetFitModel, Trainer, TrainingArguments
import torch
import gc

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    torch.cuda.empty_cache()
else:
    gpu_name = "CPU"
    print("⚠️  No GPU — this will be very slow")

gc.collect()

MODEL_DIR = f"./setfit-hts-{TRAINING_LEVEL}"

# Initialize model
print(f"\nInitializing SetFit model for '{TRAINING_LEVEL}' prediction...")
model = SetFitModel.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2",
    labels=list(set(train_dataset['label']))
)

# T4-safe settings: small batch, few contrastive iterations
args = TrainingArguments(
    batch_size=4,           # T4 needs small batches
    num_epochs=2,           # 2 epochs is enough for heading-level
    evaluation_strategy="no",  # Skip eval to save RAM
    save_strategy="epoch",
    output_dir=MODEL_DIR,
    logging_steps=50,
    num_iterations=5,       # Low iteration count prevents OOM during pair generation
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
)

print(f"\nStarting training on {gpu_name}...")
print(f"Samples: {len(train_dataset)} | Batch: 4 | Epochs: 2 | Iterations: 5")
print(f"If this OOMs, reduce MAX_TRAIN_SAMPLES in the cell above.\n")
trainer.train()

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\n✅ Training complete!")

## Step 5: Evaluate Model

In [ ]:
import time
from sklearn.metrics import accuracy_score

print("Evaluating on test set...")
start_time = time.time()

predictions = model.predict(test_dataset['text'])
accuracy = accuracy_score(test_dataset['label'], predictions)
inference_time = (time.time() - start_time) / len(test_dataset) * 1000

print(f"\n{'='*60}")
print(f"Test Accuracy: {accuracy*100:.2f}%")
print(f"Avg Inference Time: {inference_time:.2f}ms per sample")
print(f"{'='*60}")

metrics = {
    'accuracy': accuracy,
    'total_samples': len(test_dataset),
    'inference_time_ms': inference_time,
    'model_name': f'setfit-hts-{TRAINING_LEVEL}',
    'training_level': TRAINING_LEVEL,
    'training_samples': len(train_dataset),
}

import os
os.makedirs(MODEL_DIR, exist_ok=True)
with open(f'{MODEL_DIR}/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"\nMetrics saved to {MODEL_DIR}/metrics.json")

## Step 6: Test Sample Predictions

In [ ]:
# Test on some examples
test_examples = [
    "cotton t-shirt",
    "plastic water bottle",
    "leather wallet",
    "stainless steel kitchen knife",
    "wooden dining table"
]

print("Sample predictions:\n")
for example in test_examples:
    pred = model.predict([example])[0]
    print(f"'{example}' → {pred}")

## Step 7: Save and Download Model

In [ ]:
print(f"Saving model to {MODEL_DIR}...")
model.save_pretrained(MODEL_DIR)
print("Model saved!")

zip_name = f"setfit-hts-{TRAINING_LEVEL}.zip"
!zip -r {zip_name} {MODEL_DIR}/

print(f"\nDownloading {zip_name}...")
files.download(zip_name)

print(f"\n✅ Done! Extract to models/ folder and start the inference server:")
print(f"   python scripts/setfit-inference-server.py --model models/setfit-hts-{TRAINING_LEVEL}")

## Next Steps

1. Download the zip file
2. Extract it to your project: `models/setfit-hts-heading/`
3. Start the inference server:
   ```bash
   python scripts/setfit-inference-server.py --model models/setfit-hts-heading --port 5001
   ```
4. The V10 engine's heading classifier will automatically use SetFit as Tier 2